<div style="text-align: justify; max-width: 90ch">

# Lecture 01d: LangChain chains

**Status: Mandatory reading.**

In `lec_01c` you wrote `build_prompt`, `ask`, and `tag_question` by hand. Every LLM application has those same three parts, so libraries exist that name them and give them one common interface.
**LangChain** is the most widely used one, and lecture 2 builds the retrieval system on it.

**What this notebook covers**

- LangChain's parts: chat model, prompt template, output parser, and the `|` that chains them.
- `invoke` for one input, `batch` for many, structured output inside a chain.
- Reading the prompt a template renders, and swapping the local model for a hosted one in one line.

</div>

In [1]:
import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

In [2]:
# --- Course configuration: edit TIER only; settings in .env (01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

import os
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
load_dotenv(REPO / ".env")  # your settings file: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-01-basics"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. Why a framework

Your `ask` function from `lec_01c` took a prompt, called Ollama, and returned text. That is a **chat model** in LangChain:
an object with an `invoke` method. The other parts have names too:

| LangChain name | What you wrote in `lec_01c` | Job |
| --- | --- | --- |
| **chat model** (`ChatOllama`) | `client.chat(...)` | sends messages to a model, returns a message |
| **prompt template** (`ChatPromptTemplate`) | `build_prompt(question, style)` | fills named slots into messages |
| **output parser** (`StrOutputParser`) | `.message.content.strip()` | turns the model's message into what your code wants |
| **chain** (built with `\|`) | calling them in order | one object with `invoke`, `batch`, `stream` |

The value is the *common interface*: a chain built on Ollama today runs on a hosted model tomorrow by changing one line,
and lecture 2's retriever plugs in the same way. The cost is one more library whose names you must learn, and which changes between versions;
this course pins LangChain 1.x in `uv.lock`.

</div>

In [3]:
llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,  # the same switch as think=False in the ollama client
    temperature=0,
    num_predict=200,
)

reply = llm.invoke("In one sentence, what is a DataFrame?")
print(reply.content)

A DataFrame is a two-dimensional data structure used in Python's pandas library to store and manipulate tabular data, such as rows and columns, with support for various data types and operations.


In [4]:
reply.usage_metadata  # tokens in and out, the same numbers lec_01a computed by hand

{'input_tokens': 25, 'output_tokens': 38, 'total_tokens': 63}

<div style="text-align: justify; max-width: 90ch">

`invoke` returns an `AIMessage`: the `assistant` role from `lec_01c`, as an object with `content` and metadata.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Prompt templates

A template holds the messages with **named slots** in curly braces. `invoke` with a dict fills the slots and returns the rendered messages,
without calling any model, so you can look at exactly what will be sent.

</div>

In [5]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a teaching assistant for a Python course. Answer in at most {sentences} sentences.",
        ),
        ("human", "{question}"),
    ]
)

rendered = prompt.invoke(
    {"sentences": 2, "question": "What does the axis argument mean in pandas drop()?"}
)
for message in rendered.messages:
    print(f"[{message.type}] {message.content}")

[system] You are a teaching assistant for a Python course. Answer in at most 2 sentences.
[human] What does the axis argument mean in pandas drop()?


<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

Read the two rendered messages. That is the prompt the model will *actually* receive: the system message with `{sentences}` filled in, then the question.
Nobody typed that text; the template produced it. When an answer looks wrong, read the rendered prompt before blaming the model.
In lecture 2, where a system has several steps, MLflow tracing records the input and output of every step the same way.

</div>

<div style="text-align: justify; max-width: 90ch">

Curly braces are the slot syntax, which bites the first time a template contains literal braces, for example a JSON example for the model.

</div>

In [6]:
# WRONG: the JSON example's braces are read as slots named '"category"'
broken = ChatPromptTemplate.from_template(
    'Reply as JSON like {"category": "pandas"}. Question: {question}'
)
try:
    broken.invoke({"question": "How do I merge tables?"})
except KeyError as error:
    print("KeyError:", error)

KeyError: 'Input to ChatPromptTemplate is missing variables {\'"category"\'}.  Expected: [\'"category"\', \'question\'] Received: [\'question\']\nNote: if you intended {"category"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"category"}}\'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT '


In [7]:
# RIGHT: double the braces you want to keep
fixed = ChatPromptTemplate.from_template(
    'Reply as JSON like {{"category": "pandas"}}. Question: {question}'
)
print(fixed.invoke({"question": "How do I merge tables?"}).messages[0].content)

Reply as JSON like {"category": "pandas"}. Question: How do I merge tables?


<div style="text-align: justify; max-width: 90ch">

## 3. Chains

`|` joins parts into a **chain**: the output of each becomes the input of the next. `prompt | llm | StrOutputParser()` reads as "fill the template, send it, keep the text".

</div>

In [8]:
chain = prompt | llm | StrOutputParser()

print(
    chain.invoke(
        {
            "sentences": 2,
            "question": "Why do we scale features after the train/test split?",
        }
    )
)

We scale features after the train/test split to ensure that features of different scales do not disproportionately influence the model. This helps improve the performance of algorithms that are sensitive to scale, such as those using Euclidean distances.


<div style="text-align: justify; max-width: 90ch">

`batch` runs the chain over many inputs. Ollama serves one request at a time on a laptop, so we cap concurrency at 1; a hosted model would take 10 in parallel with the same line.

</div>

In [9]:
questions = [
    "What is the difference between loc and iloc?",
    "When is a silhouette score useful?",
    "What does random_state do?",
]
inputs = [{"sentences": 1, "question": q} for q in questions]

answers = chain.batch(inputs, config={"max_concurrency": 1})
pd.DataFrame({"question": questions, "answer": answers})

,question,answer
0,What is the difference between loc and iloc?,"`loc` is used for label-based indexing (e.g., ..."
1,When is a silhouette score useful?,A silhouette score is useful for evaluating th...
2,What does random_state do?,The `random_state` parameter ensures reproduci...


<div style="text-align: justify; max-width: 90ch">

Structured output from `lec_01c` is a method on the chat model: `with_structured_output` takes the same JSON schema and returns a model that answers with a dict.
It slots into a chain like any other part.

</div>

In [10]:
TAG_SCHEMA = {
    "title": "ForumTag",
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["pandas", "plots", "sklearn", "python-basics", "other"],
        },
        "needs_code": {"type": "boolean"},
    },
    "required": ["category", "needs_code"],
}

tag_prompt = ChatPromptTemplate.from_template(
    "Tag this Python-course forum question.\n\nQuestion: {question}"
)
tag_chain = tag_prompt | llm.with_structured_output(TAG_SCHEMA, method="json_schema")

tag_chain.invoke({"question": "How do I change the colours in a seaborn heatmap?"})

{'category': 'python-basics', 'needs_code': True}

<div style="text-align: justify; max-width: 90ch">

## 4. Swapping the model is one line

Everything above used `ChatOllama`. A hosted model, for a job where privacy allows it and latency matters, is the same chain with a different first line:

```python
from langchain_openai import ChatOpenAI          # any OpenAI-compatible provider
llm = ChatOpenAI(model="...", base_url="...", api_key=os.environ["PROVIDER_KEY"])
chain = prompt | llm | StrOutputParser()        # unchanged
```

The optional `lec_01f` tries it against a hosted endpoint.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- LangChain names the parts you already wrote: **chat model**, **prompt template**, **output parser**; `|` chains them into one object with `invoke`, `batch`, `stream`.
- Templates render without calling the model; literal braces must be doubled.
- `with_structured_output` puts a JSON schema inside a chain.
- Read the rendered prompt before blaming the model: it is exactly what the model receives.

**Next: lecture 2**, where the model stops answering from memory and starts answering from *your* documents: embeddings, retrieval, and a hand-built RAG, with MLflow tracing to see inside each step.

</div>